<a href="https://colab.research.google.com/github/26b21a4274kiet0-dev/Loan-payment-matrix-model-/blob/main/LOAN_PAYMENT_MATRIX_MODEL_FRONT_END_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ================================================================
# LOAN PAYMENT MATRIX MODEL (F7)
# B.Tech Mathematics Mini Project
# SINGLE-PAGE PYTHON FRONTEND
# Google Colab + Python + NumPy + SymPy + Matplotlib + Gradio
# ================================================================

!pip -q install gradio

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import gradio as gr


# ================================================================
# MATHEMATICAL ASSUMPTION
# ================================================================
#
# Standard fixed-rate EMI loan is assumed.
#
# Monthly rate:
#       r = Annual Rate / (12 * 100)
#
# Loan balance recurrence:
#
#       P(n+1) = (1+r)P(n) - EMI
#
# We represent this using an augmented matrix:
#
#       [ P(n+1) ]   [ 1+r   -EMI ] [ P(n) ]
#       [    1   ] = [  0      1  ] [  1  ]
#
# Therefore:
#
#              [1+r   -EMI]
#       A  =   [ 0      1  ]
#
# Cayley-Hamilton:
#
#       A² - tr(A)A + det(A)I = 0
#
# This is used to calculate A^60.
# ================================================================


# ================================================================
# EMI
# ================================================================

def calculate_emi(principal, annual_rate, months):

    r = annual_rate / (12 * 100)

    if r == 0:
        return principal / months

    emi = (
        principal * r * (1 + r) ** months
        / ((1 + r) ** months - 1)
    )

    return emi


# ================================================================
# CAYLEY-HAMILTON MATRIX POWER
# ================================================================

def cayley_hamilton_power(A, n):

    if n == 0:
        return sp.eye(2)

    if n == 1:
        return A

    trace_A = A.trace()
    determinant_A = A.det()

    A_previous2 = sp.eye(2)
    A_previous1 = A

    for k in range(2, n + 1):

        A_current = (
            trace_A * A_previous1
            - determinant_A * A_previous2
        )

        A_previous2 = A_previous1
        A_previous1 = A_current

    return sp.simplify(A_previous1)


# ================================================================
# SINGLE LOAN CALCULATION
# ================================================================

def calculate_single_loan(
    name,
    principal,
    annual_rate,
    term
):

    principal = float(principal)
    annual_rate = float(annual_rate)
    term = int(term)

    # ------------------------------------------------------------
    # Monthly interest rate
    # ------------------------------------------------------------

    r = annual_rate / (12 * 100)

    # ------------------------------------------------------------
    # EMI
    # ------------------------------------------------------------

    emi = calculate_emi(
        principal,
        annual_rate,
        term
    )

    # ------------------------------------------------------------
    # Transition matrix
    # ------------------------------------------------------------

    A = sp.Matrix([
        [1 + r, -emi],
        [0, 1]
    ])

    # ------------------------------------------------------------
    # Trace and determinant
    # ------------------------------------------------------------

    trace_A = A.trace()
    determinant_A = A.det()

    # ------------------------------------------------------------
    # A^60 using Cayley-Hamilton
    # ------------------------------------------------------------

    A60 = cayley_hamilton_power(A, 60)

    # ------------------------------------------------------------
    # Eigenvalues
    # ------------------------------------------------------------

    eigenvalues = A.eigenvals()

    eigen_text = ""

    for value, multiplicity in eigenvalues.items():

        eigen_text += (
            f"λ = {float(sp.N(value)):.6f} "
            f"(multiplicity {multiplicity})\n"
        )

    # ------------------------------------------------------------
    # Verify Cayley-Hamilton
    # ------------------------------------------------------------

    I = sp.eye(2)

    CH_check = sp.simplify(
        A**2
        - trace_A * A
        + determinant_A * I
    )

    # ------------------------------------------------------------
    # Loan repayment simulation
    # ------------------------------------------------------------

    balance = principal

    balances = [balance]

    total_interest = 0

    payoff_month = None

    for month in range(1, term + 1):

        interest = balance * r

        principal_payment = emi - interest

        total_interest += interest

        balance = balance - principal_payment

        if balance < 0:
            balance = 0

        balances.append(balance)

        if balance <= 0.01:

            payoff_month = month

            break

    # ------------------------------------------------------------
    # Balance at month 60
    # ------------------------------------------------------------

    if len(balances) > 60:
        balance_60 = balances[60]
    else:
        balance_60 = 0

    # ------------------------------------------------------------
    # Total amount paid
    # ------------------------------------------------------------

    total_paid = emi * payoff_month

    return {
        "name": name,
        "principal": principal,
        "rate": annual_rate,
        "term": term,
        "emi": emi,
        "matrix": A,
        "A60": A60,
        "trace": trace_A,
        "determinant": determinant_A,
        "eigenvalues": eigen_text,
        "balances": balances,
        "balance_60": balance_60,
        "interest": total_interest,
        "total_paid": total_paid,
        "payoff_month": payoff_month,
        "CH_check": CH_check
    }


# ================================================================
# MAIN FRONTEND FUNCTION
# ================================================================

def run_project(

    loan1_name,
    loan1_principal,
    loan1_rate,
    loan1_term,

    loan2_name,
    loan2_principal,
    loan2_rate,
    loan2_term,

    loan3_name,
    loan3_principal,
    loan3_rate,
    loan3_term
):

    # ------------------------------------------------------------
    # Calculate all 3 loans
    # ------------------------------------------------------------

    result1 = calculate_single_loan(
        loan1_name,
        loan1_principal,
        loan1_rate,
        loan1_term
    )

    result2 = calculate_single_loan(
        loan2_name,
        loan2_principal,
        loan2_rate,
        loan2_term
    )

    result3 = calculate_single_loan(
        loan3_name,
        loan3_principal,
        loan3_rate,
        loan3_term
    )

    results = [
        result1,
        result2,
        result3
    ]

    # ============================================================
    # TEXT OUTPUT
    # ============================================================

    output = ""

    output += "=" * 80 + "\n"
    output += "           LOAN PAYMENT MATRIX MODEL - F7\n"
    output += "=" * 80 + "\n\n"

    output += "MATHEMATICAL MODEL\n"
    output += "-" * 80 + "\n"

    output += """
P(n+1) = (1+r)P(n) - EMI

Transition Matrix:

        [ 1+r    -EMI ]
A   =   [  0       1  ]

Cayley-Hamilton theorem:

A² - tr(A)A + det(A)I = 0

A^60 is calculated using the Cayley-Hamilton recurrence.
"""

    # ============================================================
    # EACH LOAN
    # ============================================================

    for i, result in enumerate(results, 1):

        output += "\n\n"
        output += "=" * 80 + "\n"
        output += f"LOAN {i}: {result['name']}\n"
        output += "=" * 80 + "\n"

        output += (
            f"Principal Amount       : "
            f"₹{result['principal']:,.2f}\n"
        )

        output += (
            f"Annual Interest Rate   : "
            f"{result['rate']:.2f}%\n"
        )

        output += (
            f"Loan Term              : "
            f"{result['term']} months\n"
        )

        output += (
            f"Monthly EMI            : "
            f"₹{result['emi']:,.2f}\n"
        )

        # --------------------------------------------------------
        # Matrix
        # --------------------------------------------------------

        output += "\nTRANSITION MATRIX A\n"
        output += "-" * 40 + "\n"

        output += str(result["matrix"])

        # --------------------------------------------------------
        # A60
        # --------------------------------------------------------

        output += "\n\nA^60 USING CAYLEY-HAMILTON\n"
        output += "-" * 40 + "\n"

        output += str(result["A60"])

        # --------------------------------------------------------
        # Eigenvalues
        # --------------------------------------------------------

        output += "\n\nEIGENVALUES\n"
        output += "-" * 40 + "\n"

        output += result["eigenvalues"]

        # --------------------------------------------------------
        # Results
        # --------------------------------------------------------

        output += "\nLOAN RESULTS\n"
        output += "-" * 40 + "\n"

        output += (
            f"Balance after 60 months : "
            f"₹{result['balance_60']:,.2f}\n"
        )

        output += (
            f"Total Interest Paid     : "
            f"₹{result['interest']:,.2f}\n"
        )

        output += (
            f"Total Amount Paid       : "
            f"₹{result['total_paid']:,.2f}\n"
        )

        output += (
            f"Loan Paid Off At        : "
            f"Month {result['payoff_month']}\n"
        )

        # --------------------------------------------------------
        # CH verification
        # --------------------------------------------------------

        output += "\nCAYLEY-HAMILTON VERIFICATION\n"
        output += "-" * 40 + "\n"

        if result["CH_check"] == sp.zeros(2):

            output += "✓ A² - tr(A)A + det(A)I = Zero Matrix\n"
            output += "✓ Cayley-Hamilton theorem verified.\n"

        else:

            output += "✗ Verification failed.\n"


    # ============================================================
    # SUMMARY TABLE
    # ============================================================

    output += "\n\n"
    output += "=" * 100 + "\n"
    output += "                         SUMMARY\n"
    output += "=" * 100 + "\n"

    output += (
        f"{'Loan':<20}"
        f"{'EMI':>15}"
        f"{'Interest':>18}"
        f"{'Balance @60':>18}"
        f"{'Payoff':>12}\n"
    )

    output += "-" * 100 + "\n"

    for result in results:

        output += (
            f"{result['name']:<20}"
            f"₹{result['emi']:>13,.2f}"
            f"₹{result['interest']:>16,.2f}"
            f"₹{result['balance_60']:>16,.2f}"
            f"{result['payoff_month']:>12}\n"
        )

    # ============================================================
    # CHART 1: BALANCE
    # ============================================================

    fig_balance, ax = plt.subplots(
        figsize=(10, 5)
    )

    for result in results:

        ax.plot(
            range(len(result["balances"])),
            result["balances"],
            linewidth=2,
            label=result["name"]
        )

    ax.set_title(
        "Loan Balance vs Month",
        fontsize=16
    )

    ax.set_xlabel(
        "Month"
    )

    ax.set_ylabel(
        "Remaining Principal (₹)"
    )

    ax.grid(True, alpha=0.3)

    ax.legend()

    plt.tight_layout()


    # ============================================================
    # CHART 2: INTEREST
    # ============================================================

    fig_interest, ax = plt.subplots(
        figsize=(9, 5)
    )

    names = [
        result["name"]
        for result in results
    ]

    interests = [
        result["interest"]
        for result in results
    ]

    ax.bar(
        names,
        interests
    )

    ax.set_title(
        "Total Interest Paid",
        fontsize=16
    )

    ax.set_xlabel(
        "Loan Type"
    )

    ax.set_ylabel(
        "Interest (₹)"
    )

    ax.grid(
        axis="y",
        alpha=0.3
    )

    plt.xticks(
        rotation=15
    )

    plt.tight_layout()


    # ============================================================
    # CHART 3: EMI
    # ============================================================

    fig_emi, ax = plt.subplots(
        figsize=(9, 5)
    )

    emis = [
        result["emi"]
        for result in results
    ]

    ax.bar(
        names,
        emis
    )

    ax.set_title(
        "Monthly EMI Comparison",
        fontsize=16
    )

    ax.set_xlabel(
        "Loan Type"
    )

    ax.set_ylabel(
        "EMI (₹)"
    )

    ax.grid(
        axis="y",
        alpha=0.3
    )

    plt.xticks(
        rotation=15
    )

    plt.tight_layout()


    return (
        output,
        fig_balance,
        fig_interest,
        fig_emi
    )


# ================================================================
# GRADIO SINGLE-PAGE FRONTEND
# ================================================================

with gr.Blocks(
    title="Loan Payment Matrix Model - F7"
) as app:

    # ------------------------------------------------------------
    # HEADER
    # ------------------------------------------------------------

    gr.Markdown(
        """
        # 💰 Loan Payment Matrix Model
        ### B.Tech Mathematics Mini Project — F7

        **Matrix Powers • Cayley-Hamilton • Eigenvalues • EMI • Loan Repayment**
        """
    )

    gr.Markdown(
        """
        Enter the details of three loans below and click
        **Calculate Loan Model**.
        """
    )

    # ============================================================
    # INPUT SECTION
    # ============================================================

    with gr.Row():

        # --------------------------------------------------------
        # LOAN 1
        # --------------------------------------------------------

        with gr.Column():

            gr.Markdown("### 🏠 Loan 1")

            loan1_name = gr.Textbox(
                label="Loan Name",
                value="Home Loan"
            )

            loan1_principal = gr.Number(
                label="Principal (₹)",
                value=500000
            )

            loan1_rate = gr.Number(
                label="Annual Interest (%)",
                value=8.5
            )

            loan1_term = gr.Number(
                label="Term (Months)",
                value=60
            )

        # --------------------------------------------------------
        # LOAN 2
        # --------------------------------------------------------

        with gr.Column():

            gr.Markdown("### 🎓 Loan 2")

            loan2_name = gr.Textbox(
                label="Loan Name",
                value="Education Loan"
            )

            loan2_principal = gr.Number(
                label="Principal (₹)",
                value=300000
            )

            loan2_rate = gr.Number(
                label="Annual Interest (%)",
                value=10
            )

            loan2_term = gr.Number(
                label="Term (Months)",
                value=48
            )

        # --------------------------------------------------------
        # LOAN 3
        # --------------------------------------------------------

        with gr.Column():

            gr.Markdown("### 🚗 Loan 3")

            loan3_name = gr.Textbox(
                label="Loan Name",
                value="Vehicle Loan"
            )

            loan3_principal = gr.Number(
                label="Principal (₹)",
                value=800000
            )

            loan3_rate = gr.Number(
                label="Annual Interest (%)",
                value=7.5
            )

            loan3_term = gr.Number(
                label="Term (Months)",
                value=84
            )


    # ============================================================
    # CALCULATE BUTTON
    # ============================================================

    calculate_button = gr.Button(
        "🚀 CALCULATE LOAN MODEL",
        variant="primary",
        size="lg"
    )


    # ============================================================
    # OUTPUT
    # ============================================================

    gr.Markdown(
        "## 📊 Mathematical Results"
    )

    output = gr.Textbox(
        label="Results",
        lines=45,
        max_lines=60
    )


    # ============================================================
    # CHARTS
    # ============================================================

    gr.Markdown(
        "## 📈 Visual Analysis"
    )

    with gr.Row():

        balance_chart = gr.Plot(
            label="Loan Balance"
        )

        interest_chart = gr.Plot(
            label="Total Interest"
        )

        emi_chart = gr.Plot(
            label="EMI Comparison"
        )


    # ============================================================
    # BUTTON CONNECTION
    # ============================================================

    calculate_button.click(

        fn=run_project,

        inputs=[

            loan1_name,
            loan1_principal,
            loan1_rate,
            loan1_term,

            loan2_name,
            loan2_principal,
            loan2_rate,
            loan2_term,

            loan3_name,
            loan3_principal,
            loan3_rate,
            loan3_term
        ],

        outputs=[

            output,
            balance_chart,
            interest_chart,
            emi_chart
        ]
    )


# ================================================================
# LAUNCH
# ================================================================

app.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6a494f4856cafd184d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
